# Name That LLM: Exploratory Data Analysis & Stylometric Profiling

In this competition, participants must classify short passages of text (40–250 words) into one of **18 frontier language models** across 6 major AI labs.

### Evaluation Metric:
**Balanced Accuracy** (unweighted mean recall across each of the 18 model classes).

### Key Invariant:
Because test prompts are unseen, semantic and topical features can overfit. Models must instead capture **stylometric fingerprints**, token distributions, discourse markers, punctuation density, and syntactic structures.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from name_that_llm.config import TARGET_MODELS, MODEL_TO_PROVIDER
from name_that_llm.data import load_train_data
from name_that_llm.features.stylometry import StylometricExtractor

# Set aesthetic styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

## 1. Load and Inspect Training Dataset

In [ ]:
df_train = load_train_data()
print(f'Total Samples: {len(df_train)}')
print(f'Distinct Prompts: {df_train["prompt_id"].nunique()}')
print(f'Distinct Target Models: {df_train["model"].nunique()}')
df_train.head()

## 2. Target Class and Provider Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# By Provider
provider_counts = df_train["provider"].value_counts()
axes[0].bar(provider_counts.index, provider_counts.values, color=sns.color_palette('viridis', len(provider_counts)))
axes[0].set_title('Passages by AI Lab / Provider', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

# By Model
model_counts = df_train["model"].value_counts().sort_index()
axes[1].barh(model_counts.index, model_counts.values, color='steelblue')
axes[1].set_title('Passages by Target Model', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.show()

## 3. Stylometric Feature Extraction & Clustering

In [ ]:
extractor = StylometricExtractor()
feat_matrix = extractor.transform(df_train["text"])
feat_df = pd.DataFrame(feat_matrix, columns=extractor.get_feature_names_out())
feat_df["model"] = df_train["model"].values
feat_df["provider"] = df_train["provider"].values

print(f'Extracted {len(extractor.get_feature_names_out())} stylometric features.')
feat_df.head()

## 4. Key Discriminative Stylometric Signatures

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

sns.boxplot(data=feat_df, x='provider', y='rate_semicolon', ax=axes[0, 0], palette='Set2')
axes[0, 0].set_title('Semicolon Frequency (per 1,000 chars)', fontweight='bold')

sns.boxplot(data=feat_df, x='provider', y='rate_em_dash', ax=axes[0, 1], palette='Set2')
axes[0, 1].set_title('Em-Dash / Dash Frequency (per 1,000 chars)', fontweight='bold')

sns.boxplot(data=feat_df, x='provider', y='ttr_lexical_diversity', ax=axes[1, 0], palette='Set2')
axes[1, 0].set_title('Type-Token Ratio (Lexical Diversity)', fontweight='bold')

sns.boxplot(data=feat_df, x='provider', y='avg_sentence_len_words', ax=axes[1, 1], palette='Set2')
axes[1, 1].set_title('Average Sentence Length (Words)', fontweight='bold')

plt.tight_layout()
plt.show()